<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
    <p><b>Practical Homework 5 — Self-Supervised Learning (SimCLR)</b></p>
</div>

---

**Full Name:** Faraz Doagooye Tehrani

**Student ID:** 402105998

# Self-Supervised Learning with SimCLR — Homework Notebook

In this assignment you will implement **SimCLR** (*A Simple Framework for Contrastive Learning of Visual Representations*) in PyTorch, step by step.

The notebook is split into 9 sections. Each section starts with a short explanation, occasionally asks a theoretical question, and then contains one or more **TODO** code cells for you to complete. Read the comment at the top of every TODO cell carefully — it tells you exactly what is expected.

## Outline

1. **Introduction** — what is contrastive learning and why it works.
2. **Setup** — imports, device configuration, helper functions.
3. **Data Augmentation** — the SimCLR augmentation pipeline.
4. **Dataset & DataLoader** — generating two views per unlabelled image.
5. **Model Architecture** — ResNet backbone + projection head.
6. **Contrastive Loss** — the NT-Xent (InfoNCE) loss.
7. **Training Loop** — putting everything together.
8. **Linear Evaluation** — freezing the encoder and training a linear classifier on top.
9. **Discussion** — analysing the results.

> **How to submit:** Run every cell from top to bottom in order, answer all theoretical questions in markdown cells right after the question, and make sure all TODOs are completed before exporting.

## 1. Introduction

Supervised deep learning requires massive labelled datasets, which are expensive and time-consuming to create.
Self-supervised learning (SSL) is a paradigm where a model learns representations from the data itself, without external labels. One of the most successful SSL approaches is **contrastive learning**.

### The core idea

We take an image $x$ and create two random augmentations (views) $\tilde{x}_i$ and $\tilde{x}_j$.
These two views form a **positive pair** — they come from the same underlying image and should have similar feature representations.
All other images in the batch are treated as **negative examples**.

A neural network encoder $f(\cdot)$ followed by a small projection head $g(\cdot)$ maps each view to an embedding $z = g(f(\tilde{x}))$.
The loss function encourages embeddings of a positive pair to be close (similar) while pushing apart embeddings of negative examples.

### SimCLR in one paragraph

SimCLR (Chen et al., 2020) combines: (i) a strong **stochastic augmentation** module (random crop, colour jitter, random flip, Gaussian blur), (ii) a **base encoder** (ResNet-18/50), (iii) a small **projection head** (2-layer MLP), and (iv) a **contrastive loss** computed on the L2-normalised embeddings — trained with large batches and a cosine-annealing learning rate.

> **Question 1.** Why do we need a projection head $g(\cdot)$ on top of the encoder, instead of computing the loss directly on encoder features $h = f(\tilde{x})$?
>
> **Question 2.** What is the role of the temperature parameter $\tau$ in the NT-Xent loss?

**Your answer to Q1 (projection head):**

The contrastive loss doesn't need all the information in h so we use the projection to go to a space where only the contrastive loss matters and some information is lost. But when using it, we go back to h.
**Your answer to Q2 (temperature $\tau$):**

It tunes the trade off between learning from hard negatives and training stability. The higher temperature causes the model to ge more stable whereas with lower temperature it becomes sharper and better at hard negatives.

## 2. Setup

Below is the only cell in this section. The imports are already filled in (so you can focus on the SimCLR-specific parts), but you must complete two small TODOs: setting a reproducibility seed and selecting the device.

In [1]:
# -------------------- Setup --------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast

import numpy as np
import os
import sys
from tqdm import tqdm
import logging
from datetime import datetime

# TODO 2.1 — Reproducibility.
# Set the random seed for both `torch` and `numpy` to 0 so that runs are
# repeatable. (One line each.)
# Hint: torch.manual_seed(...) and np.random.seed(...)
torch.manual_seed(0)
np.random.seed(0)

# TODO 2.2 — Device selection.
# Pick CUDA if a GPU is available, otherwise CPU, and store the result in
# the variable `device` so the rest of the notebook can move tensors there.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

Using device: cuda


## 3. Data Augmentation

SimCLR uses a carefully designed augmentation pipeline that creates two *independent* random views of every image:

- **Random Resized Crop** — random crop + resize to a fixed size.
- **Random Horizontal Flip** — probability 0.5.
- **Colour Jitter** — random changes to brightness, contrast, saturation, hue (strength scaled by $s$).
- **Random Grayscale** — probability 0.2.
- **Gaussian Blur** — smooths the image with a random sigma.

### 3.1 Gaussian Blur

`torchvision` doesn't ship the SimCLR-style Gaussian blur, so we build it ourselves with two separable 1-D convolutions. Given kernel radius $r$ and sigma $\sigma$ sampled uniformly from $[0.1, 2.0]$, the 1-D weights are

$$ w_i \;\propto\; \exp\!\left(-\tfrac{i^2}{2\sigma^2}\right), \qquad i \in \{-r,\dots,r\}, $$

normalised to sum to 1. We then apply the horizontal kernel, then the vertical kernel.

> **Question 3.** Why is it important to randomly sample $\sigma$ per image rather than fixing it? What does this stochasticity buy us in contrastive learning?

**Your answer to Q3 (random $\sigma$):**

If it was constant then the model could just learn it and do a shortcut instead of learning good features and the actual semantics.

In [2]:
# -------------------- GaussianBlur --------------------
class GaussianBlur:
    """Apply a Gaussian blur with a random sigma to a PIL image (CPU)."""

    def __init__(self, kernel_size: int):
        # Force kernel_size to be odd so it has a well-defined centre.
        radius = kernel_size // 2
        kernel_size = radius * 2 + 1
        self.kernel_size = kernel_size
        self.radius = radius

        # TODO 3.1.a — Build two depthwise (groups=3) convolutions:
        #   self.blur_h : kernel shape (kernel_size, 1)    -> horizontal blur
        #   self.blur_v : kernel shape (1, kernel_size)    -> vertical blur
        # Both should have padding=0 and bias=False.
        self.blur_h = nn.Conv2d(3, 3, kernel_size=(kernel_size, 1),
                                stride=1, padding=0, bias=False, groups=3)
        self.blur_v = nn.Conv2d(3, 3, kernel_size=(1, kernel_size),
                                stride=1, padding=0, bias=False, groups=3)

        # TODO 3.1.b — Wrap reflection padding + the two conv layers in
        # a single nn.Sequential called `self.blur`.
        # Use nn.ReflectionPad2d(radius) so the output keeps the input HxW.
        self.blur = nn.Sequential(
            nn.ReflectionPad2d(radius),
            self.blur_h,
            self.blur_v,
        )

        self.pil_to_tensor = transforms.ToTensor()
        self.tensor_to_pil = transforms.ToPILImage()

    def __call__(self, img):
        # Convert PIL -> Tensor and add a batch dimension.
        img_tensor = self.pil_to_tensor(img).unsqueeze(0)

        # TODO 3.1.c — Sample a random sigma uniformly from [0.1, 2.0]
        # and build the normalised 1-D Gaussian weights of length kernel_size.
        # Steps:
        #   1) sigma = np.random.uniform(0.1, 2.0)
        #   2) x = np.arange(-self.radius, self.radius + 1)
        #   3) x = exp(-x^2 / (2 sigma^2)); then normalise so x.sum() == 1
        #   4) torch.from_numpy(x).view(1, -1).repeat(3, 1)  # one row per channel
        sigma = np.random.uniform(0.1, 2.0)
        x = np.arange(-self.radius, self.radius + 1)
        x = np.exp(-np.power(x, 2) / (2 * sigma ** 2))
        x = x / x.sum()
        x = torch.from_numpy(x).view(1, -1).repeat(3, 1)

        # Copy the 1-D kernel into the two convolution weights.
        # Expected shapes: (3, 1, kernel_size, 1) and (3, 1, 1, kernel_size).
        self.blur_h.weight.data.copy_(x.view(3, 1, self.kernel_size, 1))
        self.blur_v.weight.data.copy_(x.view(3, 1, 1, self.kernel_size))

        with torch.no_grad():
            img_tensor = self.blur(img_tensor).squeeze(0)

        return self.tensor_to_pil(img_tensor)

### 3.2 SimCLR augmentation pipeline

Now we compose the full transform. The strength factor $s$ controls the colour jitter intensity (typically $s=1$ for ImageNet-scale data, $s=0.5$ for smaller datasets).

The recommended pipeline is, in order:

1. `RandomResizedCrop(size)`
2. `RandomHorizontalFlip()`
3. `ColorJitter(0.8s, 0.8s, 0.8s, 0.2s)` wrapped in `RandomApply(p=0.8)`
4. `RandomGrayscale(p=0.2)`
5. Your custom `GaussianBlur(kernel_size = int(0.1 * size))`
6. `ToTensor()`

In [3]:
# -------------------- SimCLR transform --------------------
def get_simclr_pipeline_transform(size: int, s: float = 1.0):
    """Return a torchvision `Compose` implementing the SimCLR augmentation."""

    # TODO 3.2 — Build the colour jitter and assemble the pipeline.
    # 1) color_jitter = transforms.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
    # 2) Compose the 6 steps listed in the markdown above, in order.
    #    Apply the colour jitter with RandomApply(p=0.8).
    color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
    data_transforms = transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        GaussianBlur(kernel_size=int(0.1 * size)),
        transforms.ToTensor(),
    ])
    return data_transforms

### 3.3 Contrastive-learning view generator

For every image we need **two** augmented views (the positive pair). We wrap the transform in a tiny class whose `__call__` applies the base transform `n_views` times to the *same* PIL input and returns a list of tensors.

In [4]:
# -------------------- ContrastiveLearningViewGenerator --------------------
class ContrastiveLearningViewGenerator:
    """Produce `n_views` independently augmented views of one PIL image."""

    def __init__(self, base_transform, n_views: int = 2):
        self.base_transform = base_transform
        self.n_views = n_views

    def __call__(self, x):
        # TODO 3.3 — Return a list of length self.n_views where each entry is
        # `self.base_transform(x)`. Because the transform is stochastic, the
        # two calls produce two DIFFERENT augmented views of the same image.
        # One-liner is fine (list comprehension).
        return [self.base_transform(x) for _ in range(self.n_views)]

> **Question 4.** Why must the two views be produced by **two independent** random draws of the augmentation? What would go wrong if the *same* deterministic transform were applied to both views?

**Your answer to Q4:**

Because then the two would be the same and the positive pairs dont cause the model to learn.

## 4. Dataset and DataLoader

SimCLR does not need labels, so we use either **STL-10** (in `split='unlabeled'` mode, 96×96 images) or **CIFAR-10** (32×32 images, labels ignored).

The dataset wraps the `ContrastiveLearningViewGenerator` as its `transform`, so each sample returns `(views, _)` where `views` is a list of `n_views` tensors of shape `[C, H, W]`. The label is ignored.

We use a large batch (e.g. 256 or 512) because SimCLR benefits from many negatives, and `drop_last=True` to keep all batches the same size.

> **Question 5.** Why do we set `drop_last=True` in the `DataLoader`? What could go wrong with a final, smaller batch?

In [5]:
# -------------------- Dataset setup --------------------
class ContrastiveLearningDataset:
    def __init__(self, root_folder: str):
        self.root_folder = root_folder

    @staticmethod
    def get_simclr_pipeline_transform(size: int, s: float = 1.0):
        return get_simclr_pipeline_transform(size, s)

    def get_dataset(self, name: str, n_views: int):
        # TODO 4.1 — Fill in the two dataset factories below.
        # For 'cifar10': datasets.CIFAR10(root, train=True, download=True,
        #                                  transform = ContrastiveLearningViewGenerator(
        #                                      self.get_simclr_pipeline_transform(32), n_views))
        # For 'stl10' : datasets.STL10(root, split='unlabeled', download=True,
        #                                  transform = ContrastiveLearningViewGenerator(
        #                                      self.get_simclr_pipeline_transform(96), n_views))
        valid_datasets = {
            'cifar10': lambda: datasets.CIFAR10(
                self.root_folder, train=True, download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(32), n_views)),
            'stl10': lambda: datasets.STL10(
                self.root_folder, split='unlabeled', download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(96), n_views)),
        }
        if name not in valid_datasets:
            raise ValueError(f"Invalid dataset name: {name}. Choose 'stl10' or 'cifar10'.")
        return valid_datasets[name]()


# TODO 4.2 — Build the unlabelled training loader.
# Pick `dataset_name`, `batch_size`, `n_views`, then build the loader with:
#   shuffle=True, num_workers=2, pin_memory=True, drop_last=True
dataset_name = 'stl10'          # or 'cifar10'
batch_size = 256
n_views = 2

dataset = ContrastiveLearningDataset('./data')
train_dataset = dataset.get_dataset(dataset_name, n_views)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Number of training batches: {len(train_loader)}")
print(
    f"Each batch returns a list of {n_views} views, each of shape:",
    next(iter(train_loader))[0][0].shape,
)

100%|██████████| 2.64G/2.64G [05:22<00:00, 8.20MB/s] 


Number of training batches: 390
Each batch returns a list of 2 views, each of shape: torch.Size([256, 3, 96, 96])


**Your answer to Q5 (`drop_last=True`):**

It drops the last batch which has less negative samples and can break the algorithm.

## 5. Model Architecture

SimCLR uses a standard ResNet backbone as the encoder $f(\cdot)$ and a small MLP as the projection head $g(\cdot)$. The projection head maps the encoder output (e.g. 512-d for ResNet-18) into a lower-dimensional embedding space (e.g. 128-d). The whole model is simply $g \circ f$.

You will implement `ResNetSimCLR`:

- Take a base model name (`'resnet18'` or `'resnet50'`) and instantiate it from `torchvision.models` **without** ImageNet weights.
- Read the encoder's feature dimension from `backbone.fc.in_features` (call it `dim_mlp`).
- **Replace** `backbone.fc` with a 2-layer projection head:
  $$\texttt{Linear}(\text{dim\_mlp} \to \text{dim\_mlp}) \to \texttt{ReLU} \to \texttt{Linear}(\text{dim\_mlp} \to \text{out\_dim})$$

After contrastive pre-training the projection head is discarded; only the encoder is kept for downstream tasks.

> **Question 6.** Why does the projection head use the same dimension for its hidden layer as the encoder output? (Hint: capacity vs. bottleneck.)

**Your answer to Q6 (projection-head hidden dimension):**

If the hidden layer were narrower than the encoder output, it would create a bottleneck that irreversibly discards information in the very first linear map, limiting what the head can compute and forcing part of the invariance-related information loss back into the encoder.

In [6]:
# -------------------- ResNetSimCLR --------------------
class ResNetSimCLR(nn.Module):
    def __init__(self, base_model: str = 'resnet18', out_dim: int = 128):
        super().__init__()

        # TODO 5.1 — Pick the backbone from torchvision.models.
        # Allowed names: 'resnet18' or 'resnet50'.
        # Use num_classes=1000 (the original fc is replaced anyway) and
        # pretrained=False (we want randomly initialised weights).
        self.backbone = self._get_basemodel(base_model)

        # Read the hidden dimension that feeds into the original fc layer.
        dim_mlp = self.backbone.fc.in_features

        # TODO 5.2 — Replace `self.backbone.fc` with the SimCLR projection head:
        #     Linear(dim_mlp -> dim_mlp) -> ReLU -> Linear(dim_mlp -> out_dim)
        self.backbone.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim),
        )

    def _get_basemodel(self, model_name: str):
        resnet_models = {
            'resnet18': models.resnet18(pretrained=False, num_classes=1000),
            'resnet50': models.resnet50(pretrained=False, num_classes=1000),
        }
        if model_name not in resnet_models:
            raise ValueError(f"Invalid backbone: {model_name}. Choose 'resnet18' or 'resnet50'.")
        return resnet_models[model_name]

    def forward(self, x):
        # The forward pass just runs the modified ResNet.
        return self.backbone(x)


# Sanity check — shape should be [2, 128] for STL-10-sized inputs.
model = ResNetSimCLR(base_model='resnet18', out_dim=128).to(device)
dummy = torch.randn(2, 3, 96, 96).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Output shape: torch.Size([2, 128])


> **Question 7.** When we evaluate downstream by freezing the encoder and training only a linear classifier on top, what happens to gradients through the (now removed) projection head? Why is throwing the head away a *reasonable* choice rather than a wasteful one?

**Your answer to Q7:**

It is removed from the computational graph and no gradient passes through it. The head is much smaller and specializes at contrastive loss and therefore throwing it away is better for retaining information.

## 6. Contrastive Loss — NT-Xent

SimCLR uses the **Normalised Temperature-scaled Cross Entropy** (NT-Xent / InfoNCE) loss.

For a batch of $N$ images we obtain $2N$ embeddings (two views per image). Let $z_i, z_j$ be a positive pair and $\tau$ the temperature. The cosine similarity scaled by $1/\tau$ is

$$ \mathrm{sim}(z_a, z_b) \;=\; \frac{1}{\tau}\, \frac{z_a^\top z_b}{\lVert z_a\rVert \,\lVert z_b\rVert}. $$

For each positive pair $(i,j)$:

$$ \ell(i,j) \;=\; -\log \frac{\exp(\mathrm{sim}(z_i, z_j))}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k))}. $$

The total loss is averaged over all $2N$ such terms.

In code we build the full $2N \times 2N$ similarity matrix, drop the diagonal (self-similarities), and reshape so each row has 1 positive logit + $(2N-2)$ negative logits. With the positive always at index 0 the loss reduces to plain `CrossEntropyLoss` against the target label `0`.

> **Question 8.** Show that formulating this as cross-entropy over $(2N-1)$ classes — 1 positive + $(2N-2)$ negatives — with the correct class always being the positive, is equivalent to the equation above. *Hint:* expand $-\log \mathrm{softmax}_0(\cdot)$.

**Your answer to Q8 (cross-entropy reformulation):**

For anchor $i$ with positive $j$, collect its $2N - 1$ scaled similarities (all $k \neq i$) into a logit vector and place the positive at index 0:

$$\mathbf{u} = \big[\, \mathrm{sim}(z_i, z_j),\; \mathrm{sim}(z_i, z_{k_1}),\; \dots,\; \mathrm{sim}(z_i, z_{k_{2N-2}}) \,\big], \qquad k_m \neq i, j,$$

where $\mathrm{sim}(z_a, z_b) = \frac{1}{\tau} \frac{z_a^\top z_b}{\lVert z_a \rVert \lVert z_b \rVert}$ already includes the $1/\tau$ scaling. Cross-entropy with target class $0$ is by definition

$$\mathrm{CE}(\mathbf{u}, 0) = -\log \mathrm{softmax}_0(\mathbf{u}) = -\log \frac{e^{u_0}}{\sum_{m=0}^{2N-2} e^{u_m}}.$$

Substituting the entries of $\mathbf{u}$: the numerator is $\exp(\mathrm{sim}(z_i, z_j))$, and the denominator is

$$\sum_{m=0}^{2N-2} e^{u_m} = \exp(\mathrm{sim}(z_i, z_j)) + \sum_{k \neq i, j} \exp(\mathrm{sim}(z_i, z_k)) = \sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k)),$$

because the set $\{j\} \cup \{k : k \neq i, j\}$ is exactly $\{k : k \neq i\}$ — the diagonal (self-similarity) term $k = i$ is the one excluded by the indicator, which is precisely what dropping the diagonal in code accomplishes. Hence

$$\mathrm{CE}(\mathbf{u}, 0) = -\log \frac{\exp(\mathrm{sim}(z_i, z_j))}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k))} = \ell(i, j). \;\blacksquare$$

Averaging the cross-entropy over all $2N$ rows (each anchor with its positive at column 0) therefore reproduces the NT-Xent loss exactly, which is why `nn.CrossEntropyLoss` with all-zero targets can be used directly on the `[2N, 2N-1]` logit matrix.

In [7]:
# -------------------- info_nce_loss --------------------
def info_nce_loss(features, temperature: float = 0.07):
    """
    Args:
        features: Tensor of shape [2 * batch_size, out_dim] — the concatenated
                  embeddings of both views.
        temperature: scalar tau.

    Returns:
        logits: Tensor of shape [2N, 2N-1], where column 0 is the positive.
        labels: Tensor of shape [2N], all zeros (positive at index 0).
    """
    batch_size = features.shape[0] // 2   # N
    device = features.device

    # TODO 6.1 — Build the positive-pair MASK.
    # Construct a vector  [0,1,...,N-1, 0,1,...,N-1]  on `device`.
    # Two positions belong to the same image iff their entries are equal,
    # so `labels = (idx.unsqueeze(0) == idx.unsqueeze(1)).float()` is a
    # [2N, 2N] binary matrix marking positives (including the diagonal).
    idx = torch.cat([torch.arange(batch_size), torch.arange(batch_size)], dim=0).to(device)
    labels = (idx.unsqueeze(0) == idx.unsqueeze(1)).float()

    # TODO 6.2 — L2-normalise the features so dot products = cosine similarities.
    features = F.normalize(features, dim=1)

    # TODO 6.3 — Compute the full [2N, 2N] similarity matrix (features @ features.T).
    similarity_matrix = torch.matmul(features, features.T)

    # Drop the diagonal (self-similarity) from both matrices.
    mask = torch.eye(labels.shape[0], dtype=torch.bool, device=device)
    labels = labels[~mask].view(labels.shape[0], -1)                              # [2N, 2N-1]
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)  # [2N, 2N-1]

    # TODO 6.4 — Split positives and negatives.
    # positives : similarity at the locations where the labels mask is True   -> shape [2N, 1]
    # negatives : similarity at the locations where the labels mask is False  -> shape [2N, 2N-2]
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)          # [2N, 1]
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)  # [2N, 2N-2]

    # TODO 6.5 — Concatenate so the positive sits at column 0, then divide by tau.
    logits = torch.cat([positives, negatives], dim=1)
    logits = logits / temperature

    # Target label is 0 for every row (positive is at index 0).
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
    return logits, labels

> **Question 9.** What is the effect of *lowering* the temperature $\tau$? Discuss how it changes (i) the concentration of the softmax distribution and (ii) the magnitude of the gradients with respect to hard negatives.

**Your answer to Q9:**

Lowering τ scales up the logits. The softmax becomes more concentrated, putting most of its probability on the hardest negatives. Since the gradient for each negative is proportional to its softmax probability, hard negatives get much larger gradients, so a small τ effectively mines hard negatives. If τ is too small, training becomes unstable and semantically similar negatives are penalized too harshly.

## 7. Training Loop

We now glue everything together. Per batch:

1. Concatenate the two views along dim 0 → tensor of shape `[2N, C, H, W]`.
2. Forward through the model → embeddings of shape `[2N, out_dim]`.
3. Compute `logits, labels = info_nce_loss(features, tau)`.
4. Apply `nn.CrossEntropyLoss` to obtain the scalar loss.
5. Backprop with `GradScaler` (mixed precision is optional).
6. Log loss and top-1/top-5 contrastive accuracy every `log_every_n_steps`.
7. Step the cosine-annealing scheduler after each epoch, after a 10-epoch warm-up.

You will fill in three code cells:

- **Cell 28** — the `SimCLR` trainer class.
- **Cell 29** — an `accuracy(...)` helper for top-k accuracy.
- **Cell 30** — the hyperparameter dict + actual training run.

In [8]:
# -------------------- SimCLR Trainer --------------------
class SimCLR:
    def __init__(self, model, optimizer, scheduler, args):
        self.model = model.to(args['device'])
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.args = args
        self.criterion = nn.CrossEntropyLoss().to(args['device'])
        self.scaler = GradScaler(enabled=args['fp16_precision'])

        os.makedirs('./logs', exist_ok=True)
        logging.basicConfig(
            filename=os.path.join('./logs', f'training_{datetime.now():%Y%m%d_%H%M%S}.log'),
            level=logging.DEBUG,
        )
        logging.info("SimCLR training started.")

    def info_nce_loss(self, features):
        return info_nce_loss(features, self.args['temperature'])

    def train(self, train_loader):
        n_iter = 0
        logging.info(f"Training for {self.args['epochs']} epochs.")

        for epoch_counter in range(self.args['epochs']):
            pbar = tqdm(train_loader, desc=f"Epoch {epoch_counter}")
            for images, _ in pbar:
                # TODO 7.1 — Combine the two views.
                # `images` is a list of two tensors, each [N, C, H, W].
                # Concatenate along dim 0 to get [2N, C, H, W], then move
                # the result to `self.args['device']`.
                images = torch.cat(images, dim=0).to(self.args['device'])

                with autocast(enabled=self.args['fp16_precision']):
                    # TODO 7.2 — Forward pass and loss.
                    # 1) features = self.model(images)           -> [2N, out_dim]
                    # 2) logits, labels = self.info_nce_loss(features)
                    # 3) loss = self.criterion(logits, labels)
                    features = self.model(images)
                    logits, labels = self.info_nce_loss(features)
                    loss = self.criterion(logits, labels)

                # TODO 7.3 — Backprop with the GradScaler.
                # Sequence: zero_grad, scaler.scale(loss).backward(),
                # scaler.step(optimizer), scaler.update().
                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()

                if n_iter % self.args['log_every_n_steps'] == 0:
                    top1, top5 = accuracy(logits, labels, topk=(1, 5))
                    pbar.set_postfix({
                        'Loss': f'{loss.item():.4f}',
                        'Top1': f'{top1[0].item():.2f}%',
                        'Top5': f'{top5[0].item():.2f}%',
                    })
                    logging.debug(
                        f"Iter {n_iter} | Loss {loss.item():.4f} | "
                        f"Top1 {top1[0].item():.2f}%"
                    )

                n_iter += 1

            # TODO 7.4 — After 10 warmup epochs, step the cosine scheduler
            # once per epoch.
            if epoch_counter >= 10:
                self.scheduler.step()

        logging.info("Training finished.")
        torch.save({
            'epoch': self.args['epochs'],
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }, 'simclr_checkpoint.pth')
        print("Checkpoint saved to simclr_checkpoint.pth")

In [9]:
# -------------------- Top-k accuracy helper --------------------
def accuracy(output, target, topk=(1,)):
    """Compute the top-k accuracy (as percentages) for each k in `topk`."""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        # TODO 7.5 — Get the indices of the top-`maxk` logits per row.
        # output.topk(maxk, dim=1, largest=True, sorted=True) returns (values, indices).
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        pred = pred.t()                                  # shape: [maxk, batch_size]

        # `correct` is a [maxk, batch_size] boolean tensor that marks the
        # positions where the true class appears among the top-k predictions.
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        # TODO 7.6 — For each k in topk, count rows whose top-k slice contains
        # at least one True, divide by batch_size, and multiply by 100.
        # Append each scalar (as a tensor) to `res`.
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

In [10]:
# -------------------- Training configuration and execution --------------------
# Hyperparameters. Reduce `epochs` for demonstration; SimCLR really wants 200+
# epochs to reach its quoted performance.
args = {
    'device': device,
    'epochs': 10,
    'batch_size': batch_size,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'temperature': 0.07,
    'fp16_precision': False,     # set True if your GPU supports it
    'log_every_n_steps': 50,
    'out_dim': 128,
    'n_views': 2,
}

# TODO 7.7 — Build the model, optimizer and scheduler.
# Model    : ResNetSimCLR(base_model='resnet18', out_dim=args['out_dim']).to(device)
# Optimizer: Adam, learning rate args['lr'], weight_decay args['weight_decay']
# Scheduler: CosineAnnealingLR with T_max = len(train_loader) * args['epochs'],
#            eta_min = 0, last_epoch = -1
model = ResNetSimCLR(base_model='resnet18', out_dim=args['out_dim']).to(device)
optimizer = optim.Adam(model.parameters(), lr=args['lr'],
                       weight_decay=args['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(train_loader) * args['epochs'],
    eta_min=0,
    last_epoch=-1,
)

# TODO 7.8 — Instantiate the trainer and start training.
simclr = SimCLR(model=model, optimizer=optimizer, scheduler=scheduler, args=args)
simclr.train(train_loader)   # uncomment to actually train

/tmp/ipykernel_58/374956957.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=args['fp16_precision'])
Epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]/tmp/ipykernel_58/374956957.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args['fp16_precision']):
Epoch 9: 100%|██████████| 390/390 [06:06<00:00,  1.06it/s, Loss=1.8198, Top1=67.97%, Top5=77.73%]


Checkpoint saved to simclr_checkpoint.pth


> **Question 10.** Why is a *cosine-annealing* learning rate schedule used (after a short warm-up) instead of a constant LR? What would change qualitatively in the loss curves if we kept LR constant?

**Your answer to Q10:**

A large learning rate early on helps the model make fast progress and explore. Cosine annealing then decays it smoothly toward zero so the optimizer can settle into a good minimum instead of bouncing around. The short warmup avoids instability from the first few large batch steps. With a constant learning rate the loss drops fast at first but plateaus higher and stays noisy, because the large batch gradient noise keeps knocking the model out of narrow minima, so the final representations are worse.

## 8. Linear Evaluation

To assess the *quality* of the learned representations we follow the standard **linear-evaluation protocol**:

1. Load the pre-trained SimCLR checkpoint (`simclr_checkpoint.pth`).
2. Discard the projection head and bolt on a fresh **linear classifier** that maps the encoder output to `num_classes`.
3. **Freeze** all parameters of the backbone — only the new linear layer is trained.
4. Train on the labelled training split (STL-10 `train` or CIFAR-10) with `CrossEntropyLoss`.
5. Report top-1 and top-5 accuracy on the held-out test set.

High accuracy with a *frozen* encoder indicates that the contrastive objective produced semantically useful features.

In [11]:
# -------------------- Linear Evaluation --------------------
# Step 1 — labelled splits for the chosen dataset.
if dataset_name == 'stl10':
    train_data = datasets.STL10('./data', split='train', download=True,
                                transform=transforms.ToTensor())
    test_data  = datasets.STL10('./data', split='test',  download=True,
                                transform=transforms.ToTensor())
    num_classes = 10
elif dataset_name == 'cifar10':
    train_data = datasets.CIFAR10('./data', train=True,  download=True,
                                  transform=transforms.ToTensor())
    test_data  = datasets.CIFAR10('./data', train=False, download=True,
                                  transform=transforms.ToTensor())
    num_classes = 10
else:
    raise ValueError("Unsupported dataset for evaluation")

eval_train_loader = DataLoader(train_data, batch_size=256, shuffle=True,  num_workers=2)
eval_test_loader  = DataLoader(test_data,  batch_size=512, shuffle=False, num_workers=2)


# Step 2 — load encoder, drop the projection head, attach a linear classifier.
def load_model_for_evaluation(checkpoint_path, base_model='resnet18', num_classes=10):
    model = ResNetSimCLR(base_model=base_model, out_dim=128)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['state_dict']

    # TODO 8.1 — Strip the projection-head weights from `state_dict`.
    # The projection head sits inside `backbone.fc.*` so delete every key
    # that startswith('backbone.fc').
    for k in list(state_dict.keys()):
        if k.startswith('backbone.fc'):
            del state_dict[k]

    log = model.load_state_dict(state_dict, strict=False)
    print("Missing keys when loading backbone:", log.missing_keys)

    # TODO 8.2 — Replace the (now random) projection head with a single
    # nn.Linear from dim_mlp -> num_classes.
    dim_mlp = model.backbone.fc[0].in_features
    model.backbone.fc = nn.Linear(dim_mlp, num_classes)

    return model.to(device)


eval_model = load_model_for_evaluation('simclr_checkpoint.pth', 'resnet18', num_classes)

# TODO 8.3 — Freeze every parameter EXCEPT the new linear classifier.
# Hint: iterate over `eval_model.named_parameters()` and set
# `param.requires_grad = False` unless the name contains 'backbone.fc'.
for name, param in eval_model.named_parameters():
    if 'backbone.fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_params = [n for n, p in eval_model.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable_params)   # expect just the new fc weight+bias

eval_optimizer = optim.Adam(eval_model.parameters(), lr=1e-3, weight_decay=1e-6)
eval_criterion = nn.CrossEntropyLoss()


# TODO 8.4 — Train the linear head and evaluate after every epoch.
# For each epoch:
#   * loop over `eval_train_loader`, do a standard supervised optim step
#   * loop over `eval_test_loader` with no_grad and compute top-1 / top-5
#   * print "Epoch {e}: Loss=... | Test Top1=... | Test Top5=..."
eval_epochs = 20
for epoch in range(eval_epochs):
    # ---- train the linear head ----
    eval_model.train()
    running_loss = 0.0
    for x, y in eval_train_loader:
        x, y = x.to(device), y.to(device)
        logits = eval_model(x)
        loss = eval_criterion(logits, y)

        eval_optimizer.zero_grad()
        loss.backward()
        eval_optimizer.step()
        running_loss += loss.item()
    running_loss /= len(eval_train_loader)

    # ---- evaluate on the test set ----
    eval_model.eval()
    top1_sum, top5_sum, n_batches = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y in eval_test_loader:
            x, y = x.to(device), y.to(device)
            logits = eval_model(x)
            top1, top5 = accuracy(logits, y, topk=(1, 5))
            top1_sum += top1[0].item()
            top5_sum += top5[0].item()
            n_batches += 1

    print(f"Epoch {epoch}: Loss={running_loss:.4f} | "
          f"Test Top1={top1_sum / n_batches:.2f}% | "
          f"Test Top5={top5_sum / n_batches:.2f}%")

Missing keys when loading backbone: ['backbone.fc.0.weight', 'backbone.fc.0.bias', 'backbone.fc.2.weight', 'backbone.fc.2.bias']
Trainable parameters: ['backbone.fc.weight', 'backbone.fc.bias']
Epoch 0: Loss=1.9952 | Test Top1=46.52% | Test Top5=93.47%
Epoch 1: Loss=1.4908 | Test Top1=52.52% | Test Top5=95.21%
Epoch 2: Loss=1.3116 | Test Top1=56.37% | Test Top5=95.82%
Epoch 3: Loss=1.2128 | Test Top1=57.36% | Test Top5=96.04%
Epoch 4: Loss=1.1663 | Test Top1=58.59% | Test Top5=96.25%
Epoch 5: Loss=1.1218 | Test Top1=59.29% | Test Top5=96.56%
Epoch 6: Loss=1.0967 | Test Top1=59.42% | Test Top5=96.91%
Epoch 7: Loss=1.0698 | Test Top1=60.27% | Test Top5=96.71%
Epoch 8: Loss=1.0517 | Test Top1=59.85% | Test Top5=96.94%
Epoch 9: Loss=1.0418 | Test Top1=60.75% | Test Top5=97.11%
Epoch 10: Loss=1.0189 | Test Top1=61.24% | Test Top5=96.92%
Epoch 11: Loss=1.0220 | Test Top1=60.84% | Test Top5=96.90%
Epoch 12: Loss=1.0040 | Test Top1=60.93% | Test Top5=97.03%
Epoch 13: Loss=1.0014 | Test Top1=61

> **Question 11.** Why do we *freeze* the backbone during linear evaluation? What would happen — methodologically and empirically — if we instead fine-tuned the whole model end-to-end with the labels?

**Your answer to Q11:**

We freeze the backbone because linear evaluation is meant to test how good the pretrained features already are. If the features are linearly separable, a simple linear classifier on top will do well, and any success is clearly due to the self supervised pretraining. If we fine tuned the whole model instead, the labels would reshape the encoder itself, so the accuracy would mix feature quality with what supervised training can learn on its own, and we could no longer isolate the effect of pretraining. Empirically full fine tuning usually gives higher absolute accuracy, but it makes the comparison between methods less meaningful, which is why frozen linear evaluation is the standard protocol.

## 9. Discussion

Reflect briefly (a few sentences each) on the following — these reflections count toward your grade.

- **Batch size.** How does the batch size affect the number of negatives and the training dynamics? At what point do you expect diminishing returns?

    In SimCLR the batch is the source of negatives, so each anchor sees 2N minus 2 negatives for a batch of N images. A larger batch gives more negatives per step, which sharpens the contrastive signal and usually improves the learned features. The returns diminish once the batch already contains enough hard negatives to cover the data, so going from 256 to 4096 helps a lot but going far beyond that gives little extra while the memory and compute cost keeps rising.
- **Projection head.** Re-state, in your own words, why the projection head helps during pre-training but is discarded for downstream use.
  
  The contrastive loss forces the embeddings to become invariant to the augmentations, which throws away information like color and orientation. The projection head takes this hit so the encoder features one layer earlier can stay rich and informative. During pretraining the head shapes the space where the loss is computed, but for downstream tasks we want the fuller features, so we keep the encoder and drop the head.
- **Augmentation ablation.** Try disabling **either** colour jitter **or** Gaussian blur and re-run a short pre-training + linear-eval. Report the change in linear-eval top-1 accuracy and discuss which augmentation matters more for your dataset.

I ran it for two epochs without the gaussian blur and it got worse than that of the main one in 2 epochs by about 2 percent.
- **Random baseline.** Compare the linear-eval accuracy to that of a randomly initialised (un-pretrained) backbone with the same linear head. How big is the gap?

It got from 52 percent to 15 percent, making the gap very huge.
- **Longer training.** If you have the compute, run pre-training for 100+ epochs and report the new linear-eval numbers.

### Further Reading

- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for Contrastive Learning of Visual Representations*. ICML 2020.
- [SimCLR — official TF repo](https://github.com/google-research/simclr)
- [PyTorch SimCLR re-implementation](https://github.com/sthalles/SimCLR)
- [Exploring SimCLR — blog post](https://sthalles.github.io/simple-self-supervised-learning/)

In [13]:
# ==============================================================================
# SECTION 9: Augmentation Ablation & Random Baseline Experiments
# ==============================================================================

def get_abbreviated_simclr_transform(size: int, disable_jitter: bool = False, disable_blur: bool = False, s: float = 1.0):
    """Builds the SimCLR transform pipeline with specific augmentations disabled."""
    transform_list = [transforms.RandomResizedCrop(size=size)]
    transform_list.append(transforms.RandomHorizontalFlip())

    if not disable_jitter:
        color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
        transform_list.append(transforms.RandomApply([color_jitter], p=0.8))

    transform_list.append(transforms.RandomGrayscale(p=0.2))

    if not disable_blur:
        transform_list.append(GaussianBlur(kernel_size=int(0.1 * size)))

    transform_list.append(transforms.ToTensor())
    return transforms.Compose(transform_list)


def run_quick_pretraining(disable_jitter=False, disable_blur=False, epochs=2):
    """Runs a minimal pre-training loop for ablation studies."""
    print(f"\n--- Pre-training (Epochs: {epochs} | No Jitter: {disable_jitter} | No Blur: {disable_blur}) ---")
    
    pipe_transform = get_abbreviated_simclr_transform(
        size=96 if dataset_name == 'stl10' else 32,
        disable_jitter=disable_jitter,
        disable_blur=disable_blur
    )
    
    if dataset_name == 'stl10':
        abl_dataset = datasets.STL10('./data', split='unlabeled', download=True,
                                     transform=ContrastiveLearningViewGenerator(pipe_transform, n_views=2))
    else:
        abl_dataset = datasets.CIFAR10('./data', train=True, download=True,
                                       transform=ContrastiveLearningViewGenerator(pipe_transform, n_views=2))

    abl_loader = DataLoader(abl_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    abl_model = ResNetSimCLR(base_model='resnet18', out_dim=128).to(device)
    abl_optimizer = optim.Adam(abl_model.parameters(), lr=3e-4, weight_decay=1e-4)

    abl_model.train()
    for ep in range(epochs):
        pbar = tqdm(abl_loader, desc=f"Ablation Pre-train Epoch {ep+1}/{epochs}")
        for images, _ in pbar:
            images = torch.cat(images, dim=0).to(device)
            features = abl_model(images)
            logits, labels = info_nce_loss(features, temperature=0.07)
            loss = nn.CrossEntropyLoss()(logits, labels)

            abl_optimizer.zero_grad()
            loss.backward()
            abl_optimizer.step()
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

    return abl_model


def run_quick_linear_eval(model, epochs=3):
    """Evaluates representations by training a frozen backbone linear head."""
    for name, param in model.named_parameters():
        param.requires_grad = ('backbone.fc' in name)

    dim_mlp = model.backbone.fc[0].in_features if isinstance(model.backbone.fc, nn.Sequential) else model.backbone.fc.in_features
    model.backbone.fc = nn.Linear(dim_mlp, num_classes).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
    criterion = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        for x, y in eval_train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    top1_sum, n_batches = 0.0, 0
    with torch.no_grad():
        for x, y in eval_test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            top1, _ = accuracy(logits, y, topk=(1, 5))
            top1_sum += top1[0].item()
            n_batches += 1

    final_top1 = top1_sum / n_batches
    return final_top1


PRETRAIN_EPOCHS = 2
EVAL_EPOCHS = 3

print("Starting Ablation Study & Baseline Comparison...")

model_no_blur = run_quick_pretraining(disable_jitter=False, disable_blur=True, epochs=PRETRAIN_EPOCHS)
top1_no_blur = run_quick_linear_eval(model_no_blur, epochs=EVAL_EPOCHS)

random_model = ResNetSimCLR(base_model='resnet18', out_dim=128).to(device)
top1_random = run_quick_linear_eval(random_model, epochs=EVAL_EPOCHS)


# ------------------------------------------------------------------------------
# 3. Print Results Summary
# ------------------------------------------------------------------------------

print("\n" + "="*50)
print("              EXPERIMENT RESULTS")
print("="*50)
print(f"Random Baseline (Un-pretrained) Top-1 Accuracy : {top1_random:.2f}%")
print(f"SimCLR Without Gaussian Blur Top-1 Accuracy     : {top1_no_blur:.2f}%")
print("="*50)

Starting Ablation Study & Baseline Comparison...

--- Pre-training (Epochs: 2 | No Jitter: False | No Blur: True) ---


Ablation Pre-train Epoch 2/2: 100%|██████████| 390/390 [03:33<00:00,  1.83it/s, Loss=2.6517]



              EXPERIMENT RESULTS
Random Baseline (Un-pretrained) Top-1 Accuracy : 14.79%
SimCLR Without Gaussian Blur Top-1 Accuracy     : 50.05%
